## delong implementation

In [78]:
import pandas as pd
import numpy as np
import scipy.stats

# AUC comparison adapted from
# https://github.com/Netflix/vmaf/
def compute_midrank(x):
    """Computes midranks.
    Args:
       x - a 1D numpy array
    Returns:
       array of midranks
    """
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=np.float64)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5*(i + j - 1)
        i = j
    T2 = np.empty(N, dtype=np.float64)
    # Note(kazeevn) +1 is due to Python using 0-based indexing
    # instead of 1-based in the AUC formula in the paper
    T2[J] = T + 1
    return T2


def fastDeLong(predictions_sorted_transposed, label_1_count):
    """
    The fast version of DeLong's method for computing the covariance of
    unadjusted AUC.
    Args:
       predictions_sorted_transposed: a 2D numpy.array[n_classifiers, n_examples]
          sorted such as the examples with label "1" are first
    Returns:
       (AUC value, DeLong covariance)
    Reference:
     @article{sun2014fast,
       title={Fast Implementation of DeLong's Algorithm for
              Comparing the Areas Under Correlated Receiver Operating Characteristic Curves},
       author={Xu Sun and Weichao Xu},
       journal={IEEE Signal Processing Letters},
       volume={21},
       number={11},
       pages={1389--1393},
       year={2014},
       publisher={IEEE}
     }
    """
    # Short variables are named as they are in the paper
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=np.float64)
    ty = np.empty([k, n], dtype=np.float64)
    tz = np.empty([k, m + n], dtype=np.float64)
    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def calc_pvalue(aucs, sigma):
    """Computes log(10) of p-values.
    Args:
       aucs: 1D array of AUCs
       sigma: AUC DeLong covariances
    Returns:
       log10(pvalue)
    """
    l = np.array([[1, -1]])
    z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, sigma), l.T))
    return np.log10(2) + scipy.stats.norm.logsf(z, loc=0, scale=1) / np.log(10)


def compute_ground_truth_statistics(ground_truth):
    assert np.array_equal(np.unique(ground_truth), [0, 1])
    order = (-ground_truth).argsort()
    label_1_count = int(ground_truth.sum())
    return order, label_1_count


def delong_roc_variance(ground_truth, predictions):
    """
    Computes ROC AUC variance for a single set of predictions
    Args:
       ground_truth: np.array of 0 and 1
       predictions: np.array of floats of the probability of being class 1
    """
    order, label_1_count = compute_ground_truth_statistics(ground_truth)
    predictions_sorted_transposed = predictions[np.newaxis, order]
    aucs, delongcov = fastDeLong(predictions_sorted_transposed, label_1_count)
    assert len(aucs) == 1, "There is a bug in the code, please forward this to the developers"
    return aucs[0], delongcov


def delong_roc_test(ground_truth, predictions_one, predictions_two):
    """
    Computes log(p-value) for hypothesis that two ROC AUCs are different
    Args:
       ground_truth: np.array of 0 and 1
       predictions_one: predictions of the first model,
          np.array of floats of the probability of being class 1
       predictions_two: predictions of the second model,
          np.array of floats of the probability of being class 1
    """
    order, label_1_count = compute_ground_truth_statistics(ground_truth)
    predictions_sorted_transposed = np.vstack((predictions_one, predictions_two))[:, order]
    aucs, delongcov = fastDeLong(predictions_sorted_transposed, label_1_count)
    return aucs, calc_pvalue(aucs, delongcov)

## read files

In [151]:
# read json file from another directory
from statsmodels.stats.contingency_tables import mcnemar
import math
import numpy as np
import json
import os

def filter_out_nan(x):
    return [element for element in x if not math.isnan(element)]

def read_json_file(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

In [152]:
def getGtPreds(file1, file2):
    data1 = read_json_file(file1)
    data2 = read_json_file(file2)

    preds_member = data1['predictions']["member"]
    preds_nonmember = data1['predictions']["nonmember"]
    preds_member_ = filter_out_nan(preds_member)
    preds_nonmember_ = filter_out_nan(preds_nonmember)
    total_preds = preds_member_ + preds_nonmember_
    # While roc_auc is unaffected by which class we consider
    # positive/negative, the TPR@lowFPR calculation is.
    # Make sure the members are positive class (larger values, so negate the raw MIA scores)
    total_preds = np.array(total_preds) * -1

    preds_member2 = data2['predictions']["member"]
    preds_nonmember2 = data2['predictions']["nonmember"]
    preds_member2_ = filter_out_nan(preds_member2)
    preds_nonmember2_ = filter_out_nan(preds_nonmember2)
    total_preds2 = preds_member2_ + preds_nonmember2_
    total_preds2 = np.array(total_preds2) * -1

    
    # Assign label '0' to members for computation, since sklearn
    # expectes label '0' data to have lower values to get assigned that label
    # which is true for our attacks (lower loss for members, e.g.)
    total_labels = [1] * len(preds_member_) + [0] * len(preds_nonmember_)


    return total_labels, total_preds, total_preds2

In [159]:
attacks = ["loss", "min_k", "zlib", "ref-stablelm-base-alpha-3b-v2"]
precisions = ["8bit", "4bit", "dyn8bit"]
res = []

for precision in precisions:
    temp = []
    for attack in attacks:
        file_path1 = f"/Users/nazmul/Desktop/MIA/mimir/results_new/pythia_1.4B_org_github_experiment/EleutherAI_pythia-1.4b/github_ngram_13_<0.8_truncated/{attack}_results.json"

        file_path2 = f"/Users/nazmul/Desktop/MIA/mimir/results_new/pythia_1.4B_{precision}_github_experiment/EleutherAI_pythia-1.4b/github_ngram_13_<0.8_truncated/{attack}_results.json"

        total_labels, total_preds, total_preds2 = getGtPreds(file_path1, file_path2)
        auc, pvalue = delong_roc_test(np.array(total_labels), np.array(total_preds), np.array(total_preds2))
        print(auc)
        temp.append((pvalue[0][0]))
    res.append(temp)
# res save as csv with index
df = pd.DataFrame(res)
df.to_csv("delong_results.csv", index=False)
df

[0.697855 0.697605]
[0.699251 0.69911 ]
[0.709716 0.709568]
[0.670645 0.671379]
[0.697855 0.690582]
[0.699251 0.692988]
[0.709716 0.703631]
[0.670645 0.663409]
[0.697855 0.665813]
[0.699251 0.670475]
[0.709716 0.683819]
[0.670645 0.60917 ]


,0,1,2,3
0,-1.498706,-0.481099,-0.760901,-0.939713
1,-10.153033,-9.776383,-7.656197,-1.491510
2,-14.005024,-11.616607,-10.116620,-7.296195
